> **Niveau 🟡 moyen — les étapes sont données en commentaire, écrivez le code**

# Notebook 1 — De l'ADN du patient à la protéine

On dispose de la séquence codante de **quatre gènes exprimés dans le globule rouge**, chez un
sujet de référence et chez le patient :

| Gène | Protéine |
|---|---|
| `HBA1` | alpha-globine |
| `HBB` | bêta-globine |
| `HBD` | delta-globine |
| `HBG1` | gamma-globine (hémoglobine fœtale) |

Objectif : trouver **lequel de ces gènes est muté chez le patient**, puis suivre l'effet de
cette mutation à travers le dogme central **ADN → ARN → protéine**, jusqu'à l'acide aminé
modifié.

**Mode d'emploi**
- `Maj + Entrée` exécute une cellule et passe à la suivante.
- Exécutez les cellules **dans l'ordre**, de haut en bas.
- Les cellules **✔️ Vérification** ne se modifient pas : elles affichent ✅ quand votre code est juste.

## 0. Charger les séquences

Exécutez la cellule ci-dessous : elle crée deux fichiers FASTA — la référence et le patient —
et range leur contenu dans deux dictionnaires, `REFERENCE` et `PATIENT`.

In [ ]:
#@title ▶️ Exécutez cette cellule pour charger les séquences (ne pas modifier)
# Séquences codantes de référence (RefSeq) de quatre gènes exprimés dans le globule rouge,
# et les mêmes gènes séquencés chez le patient.

FASTA_REFERENCE = """>HBA1 alpha-globine — NM_000558.5
ATGGTGCTGTCTCCTGCCGACAAGACCAACGTCAAGGCCGCCTGGGGTAAGGTCGGCGCG
CACGCTGGCGAGTATGGTGCGGAGGCCCTGGAGAGGATGTTCCTGTCCTTCCCCACCACC
AAGACCTACTTCCCGCACTTCGACCTGAGCCACGGCTCTGCCCAGGTTAAGGGCCACGGC
AAGAAGGTGGCCGACGCGCTGACCAACGCCGTGGCGCACGTGGACGACATGCCCAACGCG
CTGTCCGCCCTGAGCGACCTGCACGCGCACAAGCTTCGGGTGGACCCGGTCAACTTCAAG
CTCCTAAGCCACTGCCTGCTGGTGACCCTGGCCGCCCACCTCCCCGCCGAGTTCACCCCT
GCGGTGCACGCCTCCCTGGACAAGTTCCTGGCTTCTGTGAGCACCGTGCTGACCTCCAAA
TACCGTTAA
>HBB bêta-globine — NM_000518.5
ATGGTGCATCTGACTCCTGAGGAGAAGTCTGCCGTTACTGCCCTGTGGGGCAAGGTGAAC
GTGGATGAAGTTGGTGGTGAGGCCCTGGGCAGGCTGCTGGTGGTCTACCCTTGGACCCAG
AGGTTCTTTGAGTCCTTTGGGGATCTGTCCACTCCTGATGCTGTTATGGGCAACCCTAAG
GTGAAGGCTCATGGCAAGAAAGTGCTCGGTGCCTTTAGTGATGGCCTGGCTCACCTGGAC
AACCTCAAGGGCACCTTTGCCACACTGAGTGAGCTGCACTGTGACAAGCTGCACGTGGAT
CCTGAGAACTTCAGGCTCCTGGGCAACGTGCTGGTCTGTGTGCTGGCCCATCACTTTGGC
AAAGAATTCACCCCACCAGTGCAGGCTGCCTATCAGAAAGTGGTGGCTGGTGTGGCTAAT
GCCCTGGCCCACAAGTATCACTAA
>HBD delta-globine — NM_000519.4
ATGGTGCATCTGACTCCTGAGGAGAAGACTGCTGTCAATGCCCTGTGGGGCAAAGTGAAC
GTGGATGCAGTTGGTGGTGAGGCCCTGGGCAGATTACTGGTGGTCTACCCTTGGACCCAG
AGGTTCTTTGAGTCCTTTGGGGATCTGTCCTCTCCTGATGCTGTTATGGGCAACCCTAAG
GTGAAGGCTCATGGCAAGAAGGTGCTAGGTGCCTTTAGTGATGGCCTGGCTCACCTGGAC
AACCTCAAGGGCACTTTTTCTCAGCTGAGTGAGCTGCACTGTGACAAGCTGCACGTGGAT
CCTGAGAACTTCAGGCTCTTGGGCAATGTGCTGGTGTGTGTGCTGGCCCGCAACTTTGGC
AAGGAATTCACCCCACAAATGCAGGCTGCCTATCAGAAGGTGGTGGCTGGTGTGGCTAAT
GCCCTGGCTCACAAGTACCATTGA
>HBG1 gamma-globine (hémoglobine fœtale) — NM_000559.3
ATGGGTCATTTCACAGAGGAGGACAAGGCTACTATCACAAGCCTGTGGGGCAAGGTGAAT
GTGGAAGATGCTGGAGGAGAAACCCTGGGAAGGCTCCTGGTTGTCTACCCATGGACCCAG
AGGTTCTTTGACAGCTTTGGCAACCTGTCCTCTGCCTCTGCCATCATGGGCAACCCCAAA
GTCAAGGCACATGGCAAGAAGGTGCTGACTTCCTTGGGAGATGCCACAAAGCACCTGGAT
GATCTCAAGGGCACCTTTGCCCAGCTGAGTGAACTGCACTGTGACAAGCTGCATGTGGAT
CCTGAGAACTTCAAGCTCCTGGGAAATGTGCTGGTGACCGTTTTGGCAATCCATTTCGGC
AAAGAATTCACCCCTGAGGTGCAGGCTTCCTGGCAGAAGATGGTGACTGCAGTGGCCAGT
GCCCTGTCCTCCAGATACCACTGA
"""

FASTA_PATIENT = """>HBA1 alpha-globine — NM_000558.5
ATGGTGCTGTCTCCTGCCGACAAGACCAACGTCAAGGCCGCCTGGGGTAAGGTCGGCGCG
CACGCTGGCGAGTATGGTGCGGAGGCCCTGGAGAGGATGTTCCTGTCCTTCCCCACCACC
AAGACCTACTTCCCGCACTTCGACCTGAGCCACGGCTCTGCCCAGGTTAAGGGCCACGGC
AAGAAGGTGGCCGACGCGCTGACCAACGCCGTGGCGCACGTGGACGACATGCCCAACGCG
CTGTCCGCCCTGAGCGACCTGCACGCGCACAAGCTTCGGGTGGACCCGGTCAACTTCAAG
CTCCTAAGCCACTGCCTGCTGGTGACCCTGGCCGCCCACCTCCCCGCCGAGTTCACCCCT
GCGGTGCACGCCTCCCTGGACAAGTTCCTGGCTTCTGTGAGCACCGTGCTGACCTCCAAA
TACCGTTAA
>HBB bêta-globine — NM_000518.5
ATGGTGCATCTGACTCCTGTGGAGAAGTCTGCCGTTACTGCCCTGTGGGGCAAGGTGAAC
GTGGATGAAGTTGGTGGTGAGGCCCTGGGCAGGCTGCTGGTGGTCTACCCTTGGACCCAG
AGGTTCTTTGAGTCCTTTGGGGATCTGTCCACTCCTGATGCTGTTATGGGCAACCCTAAG
GTGAAGGCTCATGGCAAGAAAGTGCTCGGTGCCTTTAGTGATGGCCTGGCTCACCTGGAC
AACCTCAAGGGCACCTTTGCCACACTGAGTGAGCTGCACTGTGACAAGCTGCACGTGGAT
CCTGAGAACTTCAGGCTCCTGGGCAACGTGCTGGTCTGTGTGCTGGCCCATCACTTTGGC
AAAGAATTCACCCCACCAGTGCAGGCTGCCTATCAGAAAGTGGTGGCTGGTGTGGCTAAT
GCCCTGGCCCACAAGTATCACTAA
>HBD delta-globine — NM_000519.4
ATGGTGCATCTGACTCCTGAGGAGAAGACTGCTGTCAATGCCCTGTGGGGCAAAGTGAAC
GTGGATGCAGTTGGTGGTGAGGCCCTGGGCAGATTACTGGTGGTCTACCCTTGGACCCAG
AGGTTCTTTGAGTCCTTTGGGGATCTGTCCTCTCCTGATGCTGTTATGGGCAACCCTAAG
GTGAAGGCTCATGGCAAGAAGGTGCTAGGTGCCTTTAGTGATGGCCTGGCTCACCTGGAC
AACCTCAAGGGCACTTTTTCTCAGCTGAGTGAGCTGCACTGTGACAAGCTGCACGTGGAT
CCTGAGAACTTCAGGCTCTTGGGCAATGTGCTGGTGTGTGTGCTGGCCCGCAACTTTGGC
AAGGAATTCACCCCACAAATGCAGGCTGCCTATCAGAAGGTGGTGGCTGGTGTGGCTAAT
GCCCTGGCTCACAAGTACCATTGA
>HBG1 gamma-globine (hémoglobine fœtale) — NM_000559.3
ATGGGTCATTTCACAGAGGAGGACAAGGCTACTATCACAAGCCTGTGGGGCAAGGTGAAT
GTGGAAGATGCTGGAGGAGAAACCCTGGGAAGGCTCCTGGTTGTCTACCCATGGACCCAG
AGGTTCTTTGACAGCTTTGGCAACCTGTCCTCTGCCTCTGCCATCATGGGCAACCCCAAA
GTCAAGGCACATGGCAAGAAGGTGCTGACTTCCTTGGGAGATGCCACAAAGCACCTGGAT
GATCTCAAGGGCACCTTTGCCCAGCTGAGTGAACTGCACTGTGACAAGCTGCATGTGGAT
CCTGAGAACTTCAAGCTCCTGGGAAATGTGCTGGTGACCGTTTTGGCAATCCATTTCGGC
AAAGAATTCACCCCTGAGGTGCAGGCTTCCTGGCAGAAGATGGTGACTGCAGTGGCCAGT
GCCCTGTCCTCCAGATACCACTGA
"""

with open("reference.fasta", "w") as f:
    f.write(FASTA_REFERENCE)
with open("patient.fasta", "w") as f:
    f.write(FASTA_PATIENT)


def lire_multifasta(nom_fichier):
    """Lit un fichier FASTA contenant plusieurs séquences.
    Renvoie un dictionnaire {nom du gène: séquence}."""
    sequences = {}
    nom = None
    with open(nom_fichier) as f:
        for ligne in f:
            ligne = ligne.strip()
            if ligne.startswith(">"):
                nom = ligne[1:].split()[0]
                sequences[nom] = ""
            elif ligne:
                sequences[nom] = sequences[nom] + ligne
    return sequences


REFERENCE = lire_multifasta("reference.fasta")
PATIENT = lire_multifasta("patient.fasta")
print(len(REFERENCE), "gènes chargés ✅")

Le format **FASTA** : une ligne d'en-tête qui commence par `>`, puis la séquence, coupée en
lignes de 60 lettres. Un même fichier peut contenir plusieurs séquences à la suite — ici, les
quatre gènes. Voici le fichier de référence tel qu'il est écrit sur le disque :

In [ ]:
print(FASTA_REFERENCE)

## 1. Les quatre gènes

`REFERENCE` et `PATIENT` sont des **dictionnaires** : à chaque nom de gène correspond une
séquence. Affichez le nom de chaque gène et la longueur de sa séquence.

In [ ]:
# 1. boucle for sur REFERENCE.items() : afficher le nom de chaque gène et la longueur
#    de sa séquence
# 2. afficher le nombre total de gènes
pass  # ← remplacez cette ligne par votre code

## 2. Quel gène est muté chez le patient ?

Les quatre gènes du patient ont été séquencés. Trois sont identiques à la référence, un seul
diffère — c'est celui-là qu'il faut trouver.

Deux chaînes de caractères se comparent directement : `"ACGT" == "ACGT"` vaut `True`.

In [ ]:
# 1. créer une variable gene_mute, initialisée à None
# 2. boucle for sur les noms de gènes de REFERENCE
# 3. comparer REFERENCE[nom] et PATIENT[nom] avec == : afficher "identique" ou "DIFFÉRENT"
# 4. quand les deux diffèrent, ranger le nom dans gene_mute
# 5. afficher gene_mute
# 6. définir adn_wt = REFERENCE[gene_mute] et adn_patient = PATIENT[gene_mute]
pass  # ← remplacez cette ligne par votre code

In [ ]:
# ✔️ Vérification — exécutez sans modifier
assert gene_mute == "HBB", "ce n'est pas le bon gène : recomparez chaque paire de séquences"
assert adn_wt == REFERENCE["HBB"] and adn_patient == PATIENT["HBB"], \
    "adn_wt et adn_patient doivent être les séquences du gène muté"
print("✅ le gène muté est HBB — la bêta-globine, une des deux chaînes de l'hémoglobine")

## 3. Comparer les deux séquences, base par base

À l'œil, 444 lettres, c'est trop. On écrit une fonction `trouver_mutations(ref, patient)` qui
parcourt les deux séquences position par position et renvoie la **liste des différences**,
chacune sous la forme `(index, base_ref, base_patient)`.

Exemple : `trouver_mutations("ACGT", "ACCT")` → `[(2, "G", "C")]`

⚠️ Python numérote à partir de **0** ; les biologistes numérotent les nucléotides à partir de **1**.
Le nucléotide n°1 est à l'index 0.

In [ ]:
def trouver_mutations(ref, patient):
    """Compare ref et patient position par position.
    Renvoie la liste des (index, base_ref, base_patient) où elles diffèrent."""
    # 1. créer une liste vide
    # 2. boucle for sur les index : range(len(ref))
    # 3. si ref[i] != patient[i] : ajouter (i, ref[i], patient[i]) à la liste (.append)
    # 4. renvoyer la liste
    pass  # ← remplacez cette ligne par votre code


# 5. appliquer la fonction à adn_wt et adn_patient, ranger le résultat dans `mutations`
# 6. pour chaque différence, afficher l'index Python ET le numéro du nucléotide (index + 1)

In [ ]:
# ✔️ Vérification — exécutez sans modifier
assert trouver_mutations("ACGT", "ACCT") == [(2, "G", "C")], "trouver_mutations('ACGT', 'ACCT') doit donner [(2, 'G', 'C')]"
assert trouver_mutations("AAAA", "AAAA") == [], "deux séquences identiques : la liste doit être vide"
assert len(mutations) == 1, "le patient doit avoir exactement 1 différence avec la référence"
print("✅ Mutation trouvée :", mutations)

## 4. Transcrire l'ADN en ARN

La séquence fournie est celle du **brin codant**. L'ARN messager a la même séquence que le
brin codant, à une différence près : la thymine **T** est remplacée par l'uracile **U**.

Écrivez `transcrire(adn)` qui renvoie l'ARN correspondant.

Exemple : `transcrire("ATGGAG")` → `"AUGGAG"`

In [ ]:
def transcrire(adn):
    """Renvoie l'ARN messager : la séquence de adn avec chaque T remplacé par U."""
    # indice : la méthode .replace(ancien, nouveau) des chaînes de caractères
    pass  # ← remplacez cette ligne par votre code


# transcrire adn_wt dans `arn_wt` et adn_patient dans `arn_patient`, puis afficher arn_wt

In [ ]:
# ✔️ Vérification — exécutez sans modifier
assert transcrire("ATGGAG") == "AUGGAG", "transcrire('ATGGAG') doit donner 'AUGGAG'"
assert "T" not in arn_wt and "T" not in arn_patient, "il reste des T dans l'ARN"
assert arn_wt.startswith("AUG"), "l'ARNm doit commencer par le codon start AUG"
print("✅ Transcription correcte. L'ARNm commence par", arn_wt[:3])

## 5. Découper l'ARN en codons

Le ribosome lit l'ARN **trois bases à la fois**, à partir du codon start : c'est le cadre de
lecture. Écrivez `codons(arn)` qui renvoie la liste des triplets.

Exemple : `codons("AUGGAGUAA")` → `["AUG", "GAG", "UAA"]`

Rappel : `arn[0:3]` donne les caractères d'index 0, 1 et 2.

In [ ]:
def codons(arn):
    """Découpe arn en triplets consécutifs à partir de l'index 0."""
    # 1. créer une liste vide
    # 2. boucle sur i = 0, 3, 6, ... : range(debut, fin, pas)
    # 3. ajouter à la liste la tranche arn[i:i+3]
    # 4. renvoyer la liste
    pass  # ← remplacez cette ligne par votre code


# 5. découper arn_wt dans `codons_wt` et arn_patient dans `codons_patient`
# 6. afficher le nombre de codons
# 7. index du codon muté = index du nucléotide muté (mutations[0][0]) divisé par 3 (division entière //)
#    ranger dans `index_codon`, puis afficher le numéro du codon (index + 1)
#    et le codon muté chez WT et chez le patient

In [ ]:
# ✔️ Vérification — exécutez sans modifier
assert codons("AUGGAGUAA") == ["AUG", "GAG", "UAA"], "codons('AUGGAGUAA') doit donner ['AUG', 'GAG', 'UAA']"
assert len(codons_wt) == 148, "l'ARN de HBB contient 148 codons"
assert index_codon == 6, "la mutation est dans le 7e codon, donc à l'index 6"
assert codons_wt[index_codon] == "GAG" and codons_patient[index_codon] == "GUG"
print("✅ Codon muté :", codons_wt[index_codon], "→", codons_patient[index_codon])

## 6. Traduire l'ARN en protéine

Voici le code génétique sous forme de dictionnaire : à chaque codon, il associe un acide aminé
(code à une lettre). Les trois codons stop sont notés `"*"`. Exécutez la cellule.

In [ ]:
CODE_GENETIQUE = {
    "UUU": "F", "UUC": "F", "UUA": "L", "UUG": "L",
    "UCU": "S", "UCC": "S", "UCA": "S", "UCG": "S",
    "UAU": "Y", "UAC": "Y", "UAA": "*", "UAG": "*",
    "UGU": "C", "UGC": "C", "UGA": "*", "UGG": "W",
    "CUU": "L", "CUC": "L", "CUA": "L", "CUG": "L",
    "CCU": "P", "CCC": "P", "CCA": "P", "CCG": "P",
    "CAU": "H", "CAC": "H", "CAA": "Q", "CAG": "Q",
    "CGU": "R", "CGC": "R", "CGA": "R", "CGG": "R",
    "AUU": "I", "AUC": "I", "AUA": "I", "AUG": "M",
    "ACU": "T", "ACC": "T", "ACA": "T", "ACG": "T",
    "AAU": "N", "AAC": "N", "AAA": "K", "AAG": "K",
    "AGU": "S", "AGC": "S", "AGA": "R", "AGG": "R",
    "GUU": "V", "GUC": "V", "GUA": "V", "GUG": "V",
    "GCU": "A", "GCC": "A", "GCA": "A", "GCG": "A",
    "GAU": "D", "GAC": "D", "GAA": "E", "GAG": "E",
    "GGU": "G", "GGC": "G", "GGA": "G", "GGG": "G",
}

print(CODE_GENETIQUE["AUG"], CODE_GENETIQUE["GAG"], CODE_GENETIQUE["UAA"])

Écrivez `traduire(arn)` : pour chaque codon, on ajoute l'acide aminé correspondant à la
protéine ; **au premier codon stop, on s'arrête** (le stop n'est pas ajouté).

Exemple : `traduire("AUGGAGUAAGGG")` → `"ME"`

Utilisez votre fonction `codons()` de l'exercice 4.

In [ ]:
def traduire(arn):
    """Traduit arn en protéine (code à une lettre), jusqu'au premier codon stop exclu."""
    # 1. partir d'une chaîne vide
    # 2. pour chaque codon de codons(arn) :
    #      chercher son acide aminé dans CODE_GENETIQUE
    #      si c'est "*" : sortir de la boucle (break)
    #      sinon : l'ajouter au bout de la protéine
    # 3. renvoyer la protéine
    pass  # ← remplacez cette ligne par votre code


# 4. traduire arn_wt dans `proteine_wt` et arn_patient dans `proteine_patient`
# 5. afficher proteine_wt et sa longueur

In [ ]:
# ✔️ Vérification — exécutez sans modifier
assert traduire("AUGGAGUAAGGG") == "ME", "traduire('AUGGAGUAAGGG') doit donner 'ME' (arrêt au stop)"
assert proteine_wt.startswith("MVHLTPEEK"), "la bêta-globine commence par MVHLTPEEK"
assert len(proteine_wt) == 147, "la bêta-globine compte 147 acides aminés (méthionine initiale comprise)"
assert len(proteine_patient) == len(proteine_wt), "les deux protéines ont la même longueur"
print("✅ Traduction correcte :", len(proteine_wt), "acides aminés")

## 7. Quel acide aminé a changé ?

Une protéine, comme l'ADN, est une chaîne de caractères : la fonction `trouver_mutations()`
de l'exercice 2 marche donc aussi sur les protéines.

Le dictionnaire `NOMS` donne, pour chaque lettre, l'abréviation à trois lettres et le nom.

In [ ]:
NOMS = {
    "A": ("Ala", "alanine"),
    "R": ("Arg", "arginine"),
    "N": ("Asn", "asparagine"),
    "D": ("Asp", "aspartate"),
    "C": ("Cys", "cystéine"),
    "Q": ("Gln", "glutamine"),
    "E": ("Glu", "glutamate"),
    "G": ("Gly", "glycine"),
    "H": ("His", "histidine"),
    "I": ("Ile", "isoleucine"),
    "L": ("Leu", "leucine"),
    "K": ("Lys", "lysine"),
    "M": ("Met", "méthionine"),
    "F": ("Phe", "phénylalanine"),
    "P": ("Pro", "proline"),
    "S": ("Ser", "sérine"),
    "T": ("Thr", "thréonine"),
    "W": ("Trp", "tryptophane"),
    "Y": ("Tyr", "tyrosine"),
    "V": ("Val", "valine"),
}

print(NOMS["M"])

In [ ]:
# 1. appliquer trouver_mutations à proteine_wt et proteine_patient → `differences`
# 2. pour chaque (index, aa_wt, aa_patient) de differences :
#      afficher « position N : X → Y » avec N = index + 1
#      afficher le nom complet des deux acides aminés avec NOMS[lettre][1]

In [ ]:
# ✔️ Vérification — exécutez sans modifier
assert differences == [(6, "E", "V")], "une seule différence attendue, à l'index 6 : E → V"
print("✅ Diagnostic :", NOMS["E"][0], "→", NOMS["V"][0], "en position 7 de la chaîne traduite")

## Bilan

| Niveau | Référence (WT) | Patient |
|---|---|---|
| Gène muté parmi les quatre | — | `HBB`, la bêta-globine |
| ADN, nucléotide n°20 | `A` | `T` |
| ARN, codon n°7 | `GAG` | `GUG` |
| Protéine, acide aminé n°7 | E — glutamate | V — valine |

Une seule base modifiée sur 444 remplace un acide aminé chargé (glutamate) par un acide
aminé hydrophobe (valine).

**Une simplification à connaître.** Ici, les trois autres gènes du patient sont strictement
identiques à la référence. Chez un individu réel, chacun porterait des dizaines de variants
sans conséquence : la difficulté n'est pas de trouver *une* différence, c'est de repérer
celle qui change la protéine.

**Pourquoi la littérature médicale dit « Glu6Val » et pas « Glu7Val » ?** Dans la protéine
mature, la méthionine initiale (codée par le codon start `AUG`) est retirée. La numérotation
historique de la bêta-globine commence donc à l'acide aminé suivant : le glutamate n°7 de
notre chaîne traduite y porte le n°6.

Trois numérotations pour le même acide aminé :
- index Python : **6** ;
- position dans la chaîne traduite (méthionine comprise) : **7** ;
- position dans la protéine mature (numérotation clinique) : **6**.